# dots.tts 语音合成面板（小红书 · Colab 版）

一键启动**公网面板**：输入文字 → 选音色（预设 / 上传克隆 / 音色库）→ 选语言 → 出语音。

**面板功能：**
- ✅ 界面与语言选项**全中文**
- ✅ **音色预设**：内置 4 个中文音色，点「试听」可预览
- ✅ **参考音频转写**：上传人声 → 自动识别文字 → 可手动更正（文字越准，克隆越像）
- ✅ **音色库**：把上传的声音保存下来，以后直接选，不用重复上传
- ✅ **音色相似度**：调节克隆相似程度
- ✅ 20+ 语言 + 中文方言口音

**模型缓存在你的 Google Drive**，下次启动不用重新下载 5GB。

**每次使用只需 3 步：**
1. 菜单「运行时 → 更改运行时类型 → GPU」
2. 跑「第 1 步」一键启动（首次约 5-8 分钟，之后快）
3. 打开打印出来的 `https://xxx.gradio.live` 公网地址

> ⚠️ 打开面板地址时要**开着梯子**（跟访问 Colab 同一个）。


## 第 0 步：确认 GPU（菜单操作，不是代码）

**运行时 → 更改运行时类型 → 硬件加速器选 GPU**，然后跑下面这格确认。


In [ ]:
!nvidia-smi


## 第 1 步：一键启动面板

跑这一格就行：自动挂载 Drive（缓存模型）→ 装环境（缺失才装，含转写组件）→ 启动面板 → 打印公网地址。


In [ ]:
import os, subprocess, time, re

PY = "/content/py311/bin/python"

# ---- 0. 挂载 Google Drive（缓存模型，下次免重下 5GB）----
CACHE = "/content/drive/MyDrive/dots_cache"
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    print("✅ Drive 已挂载，模型缓存：", CACHE)
except Exception as e:
    CACHE = None
    print("⚠️ Drive 未挂载（模型缓存在本地，下次需重下）：", e)

# ---- 1. 环境（缺失才重建，约 3-5 分钟）----
if not os.path.exists(PY):
    print("🔄 环境缺失，重建中（约 3-5 分钟）...")
    subprocess.run("pip install -q uv", shell=True)
    subprocess.run("uv python install 3.11", shell=True)
    subprocess.run("uv venv /content/py311 --python 3.11", shell=True)
    subprocess.run("uv pip install --python /content/py311/bin/python torch==2.11.0 torchaudio==2.11.0", shell=True)
    subprocess.run("uv pip install --python /content/py311/bin/python dots.tts huggingface_hub soundfile 'gradio==4.44.1' faster-whisper", shell=True)
    print("✅ 环境重建完成")
else:
    # 已有环境：补齐转写组件 + 固定 gradio 版本（已满足则秒过）
    subprocess.run("pip install -q uv", shell=True)
    subprocess.run("uv pip install --python /content/py311/bin/python faster-whisper 'gradio==4.44.1'", shell=True)
    print("✅ 环境已就绪（含参考音频转写组件）")

# ---- 2. 写面板脚本 ----
panel_code = r"""import os, json, shutil, time
import gradio as gr
import soundfile as sf
import torch
from dots_tts.runtime import DotsTtsRuntime
from dots_tts.utils.util import seed_everything

# ---------- 加载模型 ----------
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
PRECISION = "bfloat16" if cap[0] >= 8 else "float16"
print("加载模型...", flush=True)
runtime = DotsTtsRuntime.from_pretrained("dots-studio/dots.tts-soar", precision=PRECISION, optimize=False)
print("模型加载完成", flush=True)

# ---------- 目录 ----------
CACHE = os.environ.get("HF_HOME") or ("/content/drive/MyDrive/dots_cache" if os.path.isdir("/content/drive/MyDrive") else None)
LIB_DIR = os.path.join(CACHE, "voice_library") if CACHE else "/content/voice_library"
PRESET_DIR = "/content/presets"
os.makedirs(LIB_DIR, exist_ok=True)
os.makedirs(PRESET_DIR, exist_ok=True)
LIB_JSON = os.path.join(LIB_DIR, "voices.json")

# ---------- 内置音色预设（运行时从 GitHub 仓库下载参考音频，避免内嵌 base64 拖慢代码页） ----------
PRESET_DEFS = [
    ("婷婷（温柔女声）", "tingting.wav"),
    ("埃迪（沉稳男声）", "eddy.wav"),
    ("美佳（甜美女声）", "meijia.wav"),
    ("桑迪（知性女声）", "sandy.wav"),
]
PRESET_TEXT = "大家好，我是你的专属语音助手。今天天气很不错，我们一起聊一聊最近发生的趣事吧。"
PRESET_BASE = "https://raw.githubusercontent.com/Evan78s/dots-tts-panel/main/presets"

PRESET_LABELS = {}
for _label, _file in PRESET_DEFS:
    PRESET_LABELS[os.path.splitext(_file)[0]] = _label

def _ensure_presets():
    import urllib.request
    for _label, _file in PRESET_DEFS:
        _path = os.path.join(PRESET_DIR, _file)
        if os.path.exists(_path) and os.path.getsize(_path) > 1000:
            continue
        try:
            urllib.request.urlretrieve(PRESET_BASE + "/" + _file, _path)
            print("下载预设音色：%s" % _label, flush=True)
        except Exception as _e:
            print("⚠️ 预设音色「%s」下载失败（仍可上传参考音频使用）：%s" % (_label, _e), flush=True)

_ensure_presets()
print("内置音色预设：", list(PRESET_LABELS.values()), flush=True)

PRESET_CHOICES = [("默认音色（不克隆）", "")] + [(lbl, key) for key, lbl in PRESET_LABELS.items()]

# ---------- 语言（全部中文显示） ----------
LANG_CHOICES = [
    ("自动检测", "auto_detect"),
    ("中文（普通话）", "ZH"),
    ("英语", "EN"),
    ("粤语", "Cantonese"),
    ("日语", "JA"),
    ("韩语", "KO"),
    ("法语", "FR"),
    ("德语", "DE"),
    ("西班牙语", "ES"),
    ("俄语", "RU"),
    ("阿拉伯语", "AR"),
    ("印地语", "HI"),
    ("葡萄牙语", "PT"),
    ("意大利语", "IT"),
    ("泰语", "TH"),
    ("越南语", "VI"),
    ("印尼语", "ID"),
    ("捷克语", "CS"),
    ("荷兰语", "NL"),
    ("芬兰语", "FI"),
    ("希腊语", "EL"),
    ("波兰语", "PL"),
    ("罗马尼亚语", "RO"),
    ("土耳其语", "TR"),
    ("乌克兰语", "UK"),
    ("口音：北京官话", "口音:北京官话"),
    ("口音：东北话", "口音:东北话"),
    ("口音：四川话", "口音:四川话"),
    ("口音：闽南话", "口音:闽南话"),
    ("口音：吴语", "口音:吴语"),
]

# ---------- 音色库（持久化到 Drive） ----------
def load_library():
    if os.path.exists(LIB_JSON):
        try:
            return json.load(open(LIB_JSON, encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_library(lib):
    json.dump(lib, open(LIB_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# ---------- 参考音频转写（ASR） ----------
_whisper = None
def get_whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        _whisper = WhisperModel(
            "small",
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="float16" if torch.cuda.is_available() else "int8",
        )
    return _whisper

def do_transcribe(ref_audio):
    if not ref_audio:
        return "", "⚠️ 请先上传参考音频。"
    try:
        model = get_whisper()
        segments, _ = model.transcribe(ref_audio, beam_size=1)
        text = "".join(s.text for s in segments).strip()
        return text, "✅ 识别完成，请核对并更正下方文字（文字越准，克隆越像）。"
    except Exception as e:
        return "", "⚠️ 识别失败：" + str(e) + "（可手动填写参考音频说了什么）"

# ---------- 音色库操作 ----------
def do_save_voice(name, ref_audio, ref_text):
    if not name or not name.strip():
        raise gr.Error("请先填写音色名称。")
    if not ref_audio:
        raise gr.Error("请先上传参考音频。")
    name = name.strip()
    lib = load_library()
    ext = os.path.splitext(ref_audio)[1].lower() or ".wav"
    dst = os.path.join(LIB_DIR, "%02d_%d%s" % (len(lib) + 1, int(time.time()), ext))
    shutil.copy(ref_audio, dst)
    lib[name] = {"file": os.path.basename(dst), "prompt_text": (ref_text or "").strip()}
    save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=name), "✅ 已保存音色「%s」到音色库（共 %d 个）。" % (name, len(choices))

def do_delete_voice(lib_voice):
    lib = load_library()
    if lib_voice in lib:
        _f = lib.pop(lib_voice)
        try:
            os.remove(os.path.join(LIB_DIR, _f["file"]))
        except Exception:
            pass
        save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=None), "已删除音色「%s」。" % lib_voice

def preview_preset(preset):
    if not preset:
        return None, "默认音色无需试听，直接合成即可。"
    path = os.path.join(PRESET_DIR, preset + ".wav")
    if not os.path.exists(path):
        return None, "⚠️ 预设音频不存在。"
    data, sr = sf.read(path)
    return (sr, data), "试听预设音色：%s" % PRESET_LABELS.get(preset, preset)

# ---------- 合成 ----------
def synth(source, preset, ref_audio, ref_text, lib_voice, synth_text, synth_lang, speaker_scale,
          seed=0, num_steps=10, guidance_scale=1.2, normalize_text=False):
    prompt_path = None
    prompt_text = None
    info = []
    if source == "音色预设":
        if preset:
            prompt_path = os.path.join(PRESET_DIR, preset + ".wav")
            prompt_text = PRESET_TEXT
            info.append("音色预设：" + PRESET_LABELS.get(preset, preset))
    elif source == "上传参考音频":
        if ref_audio:
            prompt_path = ref_audio
            prompt_text = (ref_text or "").strip() or None
            info.append("音色：上传参考音频")
    else:
        if lib_voice:
            _e = load_library().get(lib_voice)
            if _e:
                prompt_path = os.path.join(LIB_DIR, _e["file"])
                prompt_text = _e.get("prompt_text") or None
                info.append("音色库：" + lib_voice)
    if not synth_text or not synth_text.strip():
        raise gr.Error("请先输入要合成的文字。")
    lang = synth_lang or "auto_detect"
    if seed and int(seed) > 0:
        seed_everything(int(seed))
        info.append("音色种子 %d" % int(seed))
    res = runtime.generate(text=synth_text.strip(), language=lang,
                           prompt_audio_path=prompt_path, prompt_text=prompt_text,
                           speaker_scale=speaker_scale,
                           num_steps=int(num_steps), guidance_scale=float(guidance_scale),
                           normalize_text=bool(normalize_text))
    audio = res["audio"].float().cpu().squeeze().numpy()
    sr = res["sample_rate"]
    dur = round(len(audio) / sr, 2)
    info.append("语言：" + lang)
    info.append("%d 秒 · %d Hz" % (round(dur), sr))
    if prompt_path:
        info.append("音色相似度 %.1f" % speaker_scale)
    else:
        info.append("未用参考音色（模型默认声音）")
    return (sr, audio), " · ".join(info)

# ---------- 音色来源切换：控制各区域显示 ----------
def on_source_change(src):
    if src == "音色预设":
        return (gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False))
    if src == "上传参考音频":
        return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=False),
                gr.update(visible=False))
    # 音色库
    return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
            gr.update(visible=True))

# ---------- 顶部 Banner（标题 + 说明 + 联系链接） ----------
BILIBILI_URL = "https://space.bilibili.com/380877309"
DAOYAKE_URL = "https://www.daoyanke.cn"

_BANNER_HTML = (
    '<div style="text-align:center;padding:20px 14px;background:linear-gradient(135deg,#667eea,#764ba2);border-radius:14px;margin-bottom:14px;">'
    '<h1 style="color:#fff;margin:0 0 6px;font-size:28px;">🎙️ dots.tts 语音合成面板</h1>'
    '<p style="color:#eaeaff;margin:0 0 14px;font-size:15px;">输入文字 → 选音色（预设 / 上传克隆 / 音色库）→ 选语言 → 一键合成<br>支持声音克隆 · 20+ 语言 · 中文方言口音</p>'
    '<p style="margin:0;">'
    '<a href="' + BILIBILI_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">📺 B站</a>'
    '<a href="' + DAOYAKE_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">🎬 导演课</a>'
    '</p></div>'
)

# ---------- 界面 ----------
with gr.Blocks(title="dots.tts 语音合成面板") as demo:
    gr.HTML(_BANNER_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## ① 音色设置")
            source = gr.Radio(["音色预设", "上传参考音频", "音色库"], value="音色预设", label="音色来源")
            preset_dd = gr.Dropdown(PRESET_CHOICES, value="", label="音色预设")
            preview_btn = gr.Button("试听预设音色")
            preview_audio = gr.Audio(label="预设试听")
            ref_audio = gr.Audio(label="参考音频（3-10 秒清晰人声）", type="filepath", visible=False)
            transcribe_btn = gr.Button("识别转写", visible=False)
            ref_text = gr.Textbox(label="参考音频文字（自动识别，可手动更正）", lines=3, visible=False,
                                  placeholder="上传音频后点「识别转写」自动填写；也可直接手填。文字越准，克隆越像。")
            voice_name = gr.Textbox(label="音色名称（保存到音色库）", visible=False, placeholder="例如：我的声音")
            save_btn = gr.Button("保存到音色库", visible=False)
            lib_dd = gr.Dropdown(choices=list(load_library().keys()), value=None,
                                 label="音色库（已保存的音色）", visible=False)
            delete_btn = gr.Button("删除选中音色", visible=False)
            voice_status = gr.Textbox(label="提示", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("## ② 合成")
            synth_text = gr.Textbox(label="要合成的文字", lines=4, value="你好，欢迎使用 dots.tts 语音合成面板。")
            synth_lang = gr.Dropdown(LANG_CHOICES, value="ZH", label="语言")
            speaker_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.5, step=0.1,
                                      label="音色相似度（使用参考音色时生效，越高越像）")
            with gr.Accordion("⚙️ 高级设置（可选）", open=False):
                seed = gr.Slider(minimum=0, maximum=9999, value=0, step=1,
                                 label="音色种子（0=随机；固定数字=每次生成同一个声音）")
                num_steps = gr.Slider(minimum=10, maximum=32, value=10, step=1,
                                      label="生成质量·采样步数（越大越细腻，但更慢）")
                guidance_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.2, step=0.1,
                                           label="引导强度（越大越贴合文字与参考音色）")
                normalize_text = gr.Checkbox(value=False, label="文本规范化（数字/符号自动转口语读法）")
            synth_btn = gr.Button("开始合成", variant="primary")
            result_audio = gr.Audio(label="合成结果")
            result_info = gr.Textbox(label="结果信息", interactive=False)

    _voice_components = [preset_dd, preview_btn, preview_audio, ref_audio, transcribe_btn,
                         ref_text, voice_name, save_btn, lib_dd, delete_btn]
    source.change(on_source_change, source, _voice_components)
    preview_btn.click(preview_preset, preset_dd, [preview_audio, voice_status])
    transcribe_btn.click(do_transcribe, ref_audio, [ref_text, voice_status])
    save_btn.click(do_save_voice, [voice_name, ref_audio, ref_text], [lib_dd, voice_status])
    delete_btn.click(do_delete_voice, lib_dd, [lib_dd, voice_status])
    synth_btn.click(synth, [source, preset_dd, ref_audio, ref_text, lib_dd, synth_text, synth_lang,
                            speaker_scale, seed, num_steps, guidance_scale, normalize_text],
                    [result_audio, result_info])

demo.launch(share=True, debug=False)
"""
open("panel.py", "w", encoding="utf-8").write(panel_code)

# ---- 3. 启动面板 + 拿公网地址 ----
env = dict(os.environ)
if CACHE:
    env["HF_HOME"] = CACHE
subprocess.Popen([PY, "-u", "panel.py"], stdout=open("panel.log", "w"), stderr=subprocess.STDOUT, env=env)

url = None
for i in range(1, 601):
    time.sleep(1)
    if os.path.exists("panel.log"):
        m = re.search("https://[a-z0-9-]+.gradio.live", open("panel.log").read())
        if m:
            url = m.group(0)
            break
    if i % 30 == 0:
        print(f"  ... 已等 {i} 秒（首次加载模型较慢，尤其从 Drive 读取）", flush=True)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 面板公网地址：", url)
    print("   用浏览器打开这个地址即可（保持梯子开启）。")
else:
    print("⚠️ 未获取到地址，日志：")
    print(open("panel.log").read()[-2000:] if os.path.exists("panel.log") else "无日志")


## 🔄 重启面板（会话没断、但面板挂了时用）

如果 Colab 还开着、只是面板打不开/地址失效，跑这格快速重启（**不重装环境**，只重新加载模型，约 1 分钟）。


In [ ]:
import subprocess, os, time, re

PY = "/content/py311/bin/python"
CACHE = "/content/drive/MyDrive/dots_cache"

# 杀掉旧面板进程
subprocess.run("pkill -f panel.py || true", shell=True)
time.sleep(2)

# 重新启动（环境已存在，不重装）
env = dict(os.environ)
if os.path.isdir(CACHE):
    env["HF_HOME"] = CACHE
subprocess.Popen([PY, "-u", "panel.py"], stdout=open("panel.log", "w"), stderr=subprocess.STDOUT, env=env)

url = None
for i in range(1, 601):
    time.sleep(1)
    if os.path.exists("panel.log"):
        m = re.search("https://[a-z0-9-]+.gradio.live", open("panel.log").read())
        if m:
            url = m.group(0)
            break
    if i % 30 == 0:
        print(f"  ... 已等 {i} 秒", flush=True)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 新面板地址：", url)
else:
    print("⚠️ 失败，日志：")
    print(open("panel.log").read()[-2000:] if os.path.exists("panel.log") else "无日志")


## 📌 下次怎么用（重要）

**把本 notebook 保存到你的 Google Drive**，以后直接从 Drive 打开：

1. 菜单 **文件 → 在 Drive 中保存副本**
2. 下次用：从 Drive 打开这个副本 → 选 GPU → 跑「第 1 步」即可

**为什么快：**
- 模型缓存到了 Drive（`/content/drive/MyDrive/dots_cache`），**下次不重下 5GB**
- 音色库也存到 Drive（`dots_cache/voice_library/`），**保存的音色下次还在**
- 只有 Python 环境需要重装（约 2-3 分钟），这是 Colab 免费版不可避免的

**地址有效期：** 面板地址只要 Colab 会话不断线就有效；断线重连后重跑「第 1 步」会拿到新地址。
